# Task 3: AI Ethics and Fairness Check
**CM2604 Machine Learning: Coursework Part 01**

Checks whether the best model treats different customer groups fairly, and lists the ethical and post-deployment strategies.

This notebook is self-contained, so it re-trains the tuned models from Task 2 (same split, same grids, same random seed) before the fairness check.

## Setup and data loading

In [ ]:
# Import all libraries used in this notebook
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (ConfusionMatrixDisplay, RocCurveDisplay,
                             accuracy_score, classification_report,
                             f1_score, precision_score, recall_score,
                             roc_auc_score)
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

RANDOM_STATE = 42
TARGET = "Churn"

# Folders for saving figures and results (downloaded at the end)
FIG_DIR = Path("figures")
RESULTS_DIR = Path("results")
FIG_DIR.mkdir(exist_ok=True)
RESULTS_DIR.mkdir(exist_ok=True)


def save(fig, name):
    """Save a figure to figures/ and show it in the notebook."""
    fig.tight_layout()
    fig.savefig(FIG_DIR / name, dpi=150, bbox_inches="tight")
    plt.show()

The dataset is loaded from the `data/` folder of the project's public GitHub repository.
**Change `GITHUB_USERNAME` below to your own GitHub username.**

In [ ]:
GITHUB_USERNAME = "yasiru20250101"         # your GitHub username
REPO_NAME = "telco-churn-classification"
DATA_URL = (f"https://raw.githubusercontent.com/{GITHUB_USERNAME}/{REPO_NAME}"
            "/main/data/WA_Fn-UseC_-Telco-Customer-Churn.csv")

raw = pd.read_csv(DATA_URL)
print("Shape:", raw.shape)
raw.head()

## Data cleaning

In [ ]:
print("Data types:")
print(raw.dtypes)
print("\nMissing values (NaN):", raw.isna().sum().sum())

# TotalCharges is stored as text: check for blank strings
blank_tc = raw["TotalCharges"].astype(str).str.strip() == ""
print("Blank strings in TotalCharges:", blank_tc.sum())
print("Tenure of those customers:", raw.loc[blank_tc, "tenure"].unique())

In [ ]:
NUMERIC_FEATURES = ["tenure", "MonthlyCharges", "TotalCharges"]
CATEGORICAL_FEATURES = [
    "gender", "SeniorCitizen", "Partner", "Dependents", "PhoneService",
    "MultipleLines", "InternetService", "OnlineSecurity", "OnlineBackup",
    "DeviceProtection", "TechSupport", "StreamingTV", "StreamingMovies",
    "Contract", "PaperlessBilling", "PaymentMethod",
]


def clean_data(df):
    """
    1. Drop customerID (unique identifier, no predictive value).
    2. Convert TotalCharges to numeric; blanks (tenure = 0) become NaN.
    3. Fill those NaN with 0 because new customers have not been charged yet.
    4. Convert SeniorCitizen 0/1 to No/Yes so it is treated as categorical.
    5. Encode the target: Yes -> 1, No -> 0.
    6. Remove duplicate rows.
    """
    df = df.copy()
    df = df.drop(columns=["customerID"])
    df["TotalCharges"] = pd.to_numeric(df["TotalCharges"].astype(str).str.strip(),
                                       errors="coerce")
    df["TotalCharges"] = df["TotalCharges"].fillna(0)
    df["SeniorCitizen"] = df["SeniorCitizen"].map({0: "No", 1: "Yes"})
    df[TARGET] = df[TARGET].map({"No": 0, "Yes": 1})
    before = len(df)
    df = df.drop_duplicates()
    print(f"Duplicate rows removed: {before - len(df)}")
    return df


df = clean_data(raw)
print("Clean shape:", df.shape)
df[NUMERIC_FEATURES].describe().round(2)

## Re-create the tuned models from Task 2

In [ ]:
# Stratified 80/20 split keeps the same churn ratio in train and test sets
X = df.drop(columns=[TARGET])
y = df[TARGET]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE)

print("Train:", X_train.shape, "| Test:", X_test.shape)
print("Churn rate train: %.3f | test: %.3f" % (y_train.mean(), y_test.mean()))


def build_preprocessor(scale=True):
    """
    Numeric     -> StandardScaler (KNN and LR need scaling; RF does not -> scale=False)
    Categorical -> One-Hot Encoding (drop='if_binary' keeps one column for Yes/No)
    Used inside a Pipeline so it is fitted on the training data only (no leakage).
    """
    numeric_step = StandardScaler() if scale else "passthrough"
    return ColumnTransformer(transformers=[
        ("num", numeric_step, NUMERIC_FEATURES),
        ("cat", OneHotEncoder(drop="if_binary", handle_unknown="ignore"),
         CATEGORICAL_FEATURES),
    ])


n_features = len(build_preprocessor().fit(X_train).get_feature_names_out())
print("Number of features after encoding:", n_features)

In [ ]:
SCORING = "f1"
CV = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

models = {
    "KNN": (
        Pipeline([("pre", build_preprocessor(scale=True)),
                  ("clf", KNeighborsClassifier())]),
        {
            "clf__n_neighbors": [3, 5, 7, 11, 15, 21, 31],
            "clf__weights": ["uniform", "distance"],
            "clf__metric": ["euclidean", "manhattan"],
        },
    ),
    "Logistic Regression": (
        Pipeline([("pre", build_preprocessor(scale=True)),
                  ("clf", LogisticRegression(max_iter=2000,
                                             random_state=RANDOM_STATE))]),
        {
            # C = inverse of regularisation strength (L2 regularisation)
            "clf__C": [0.001, 0.01, 0.1, 1, 10, 100],
            "clf__class_weight": [None, "balanced"],
        },
    ),
    "Random Forest": (
        Pipeline([("pre", build_preprocessor(scale=False)),
                  ("clf", RandomForestClassifier(random_state=RANDOM_STATE,
                                                 n_jobs=-1))]),
        {
            "clf__n_estimators": [100, 300],
            "clf__max_depth": [None, 6, 10, 15],
            "clf__min_samples_split": [2, 10],
            "clf__min_samples_leaf": [1, 4],
            "clf__class_weight": [None, "balanced"],
        },
    ),
}


def evaluate(name, setting, model):
    """Return test-set metrics for one fitted model."""
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]
    print(f"\n===== {name} ({setting}) =====")
    print(classification_report(y_test, y_pred, target_names=["No churn", "Churn"]))
    return {
        "Model": name,
        "Setting": setting,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1": f1_score(y_test, y_pred),
        "ROC_AUC": roc_auc_score(y_test, y_prob),
    }

In [ ]:
# Tuned models only (same grids and seed as Task 2, so the results match)
results = []
tuned_models = {}
for name, (pipe, grid) in models.items():
    search = GridSearchCV(pipe, grid, scoring=SCORING, cv=CV, n_jobs=-1)
    search.fit(X_train, y_train)
    tuned_models[name] = search.best_estimator_
    y_pred = search.best_estimator_.predict(X_test)
    results.append({"Model": name, "F1": f1_score(y_test, y_pred)})
    print(f"{name}: best params {search.best_params_}")

results_df = pd.DataFrame(results).round(4)
best_name = results_df.loc[results_df["F1"].idxmax(), "Model"]
best_model = tuned_models[best_name]
print("\nBest model by F1:", best_name)
results_df

## Fairness check across sensitive groups

Does the best model perform equally well for different groups (gender, SeniorCitizen)?

In [ ]:
preds = X_test[["gender", "SeniorCitizen"]].copy()
preds["y_true"] = y_test.values
preds["y_pred"] = best_model.predict(X_test)

rows = []
for attr in ["gender", "SeniorCitizen"]:
    for group, g in preds.groupby(attr):
        rows.append({
            "Attribute": attr,
            "Group": group,
            "Count": len(g),
            "Actual churn rate": g["y_true"].mean(),
            "Predicted churn rate": g["y_pred"].mean(),
            "Recall": recall_score(g["y_true"], g["y_pred"], zero_division=0),
            "Precision": precision_score(g["y_true"], g["y_pred"], zero_division=0),
            "F1": f1_score(g["y_true"], g["y_pred"], zero_division=0),
        })

fairness = pd.DataFrame(rows).round(3)
fairness.to_csv(RESULTS_DIR / "fairness_report.csv", index=False)

# A large recall gap means the model misses churners in one group more often
for attr, g in fairness.groupby("Attribute"):
    gap = g["Recall"].max() - g["Recall"].min()
    print(f"Recall gap for {attr}: {gap:.3f} -> {'CHECK' if gap > 0.10 else 'OK'}")
fairness

In [ ]:
# Visualise recall and predicted churn rate per group
plot_df = fairness.melt(id_vars=["Attribute", "Group"],
                        value_vars=["Actual churn rate", "Predicted churn rate", "Recall"],
                        var_name="Measure", value_name="Value")
g = sns.catplot(data=plot_df, x="Group", y="Value", hue="Measure",
                col="Attribute", kind="bar", sharex=False, height=4, aspect=1.1)
g.figure.suptitle(f"Fairness check - {best_name}", y=1.04)
g.savefig(FIG_DIR / "13_fairness_check.png", dpi=150, bbox_inches="tight")
plt.show()

## Ethical strategies followed during development
- **Fairness testing:** recall and predicted churn rate compared across gender and SeniorCitizen groups (above).
- **Privacy / data minimisation:** `customerID` removed; only the public, anonymised Kaggle data used.
- **Transparency:** interpretable models (Logistic Regression coefficients, Random Forest feature importance) included.
- **No data leakage:** preprocessing inside Pipelines, held-out test set, cross-validation.
- **Reproducibility:** fixed random seed and full Git history.

## Post-deployment strategies
1. Monitor recall, precision and F1 on new customers every month.
2. Detect data drift (e.g. changes in MonthlyCharges or contract mix).
3. Retrain on recent data on a schedule and validate before release.
4. Repeat fairness audits after every retrain.
5. Keep a human in the loop: predictions support staff, never automatic penalties.
6. Use predictions only for positive actions such as retention offers.
7. Secure the model and customer data with access control and logging.
8. Document the model (model card) with intended use and limitations.

## Download figures and results

In [ ]:
# Zip the figures/ and results/ folders and download them for the report
import shutil
shutil.make_archive("task3_figures", "zip", ".", "figures")
shutil.make_archive("task3_results", "zip", ".", "results")

try:
    from google.colab import files
    files.download("task3_figures.zip")
    files.download("task3_results.zip")
except ImportError:
    print("Not running in Colab: zip files saved in the current folder.")